In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, r2_score


In [ ]:
data = pd.read_csv(r"job_salary_prediction_dataset.csv")
df = pd.DataFrame(data)
df.head()

In [ ]:
df.isnull().sum().sort_values(ascending = False)

In [ ]:
# Checking Outliers
num_cols = df.select_dtypes(include = ['number']).columns
outlier_report = {}

for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - (1.5 * IQR)
    upper = Q3 + (1.5 * IQR)
    count = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = count
print("--- Outlier Count Per Column ---")
for col, count in outlier_report.items():
    print(f"{col}: {count} outliers")




In [ ]:
# ---------------------------------------------------------
# 1. DATA LOADING & SAMPLING (For Speed)
# ---------------------------------------------------------
df = pd.read_csv('job_salary_prediction_dataset.csv')

# Taking only 10,000 sample rows
df_sample = df.sample(10000, random_state=42).copy()

print(f"Data Sampled! Working on: {df_sample.shape[0]} rows.")

# ---------------------------------------------------------
# 2. REMOVING OUTLIERS (Your IQR Logic)
# ---------------------------------------------------------
Q1 = df_sample['salary'].quantile(0.25)
Q3 = df_sample['salary'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

df_cleaned = df_sample[(df_sample['salary'] >= lower_bound) & (df_sample['salary'] <= upper_bound)].copy()

# ---------------------------------------------------------
# 3. FEATURE ENGINEERING
# ---------------------------------------------------------
df_cleaned['profile_score'] = df_cleaned['experience_years'] + df_cleaned['certifications']

# ---------------------------------------------------------
# 4. LABEL ENCODING (Categorical Columns)
# ---------------------------------------------------------
le = LabelEncoder()
cat_cols = ['job_title', 'education_level', 'industry', 'company_size', 'location', 'remote_work']

for col in cat_cols:
    df_cleaned[col] = le.fit_transform(df_cleaned[col])

# ---------------------------------------------------------
# 5. SPLITTING DATA (X and y)
# ---------------------------------------------------------
X = df_cleaned.drop('salary', axis=1)
y = df_cleaned['salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ---------------------------------------------------------
# 6. SCALING (Mandatory for KNN and SVM)
# ---------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# 7. MODEL TRAINING (Optimized Parameters)
# ---------------------------------------------------------

# A. Random Forest (Using n_jobs=-1 to use all CPU cores)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)

# B. KNN
knn = KNeighborsRegressor(n_neighbors=7)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)

# C. SVM (SVR) 
svm = SVR(kernel='rbf', C=1000)
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)

# ---------------------------------------------------------
# 8. EVALUATION (MAE & R2 Score)
# ---------------------------------------------------------
print("\n--- Model Performance Report (10k Sample) ---")
print(f"Random Forest - MAE: {mean_absolute_error(y_test, rf_pred):.2f}, R2: {r2_score(y_test, rf_pred):.4f}")
print(f"KNN           - MAE: {mean_absolute_error(y_test, knn_pred):.2f}, R2: {r2_score(y_test, knn_pred):.4f}")
print(f"SVM (SVR)     - MAE: {mean_absolute_error(y_test, svm_pred):.2f}, R2: {r2_score(y_test, svm_pred):.4f}")

# ---------------------------------------------------------
# 9. VISUALIZATION
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.scatter(y_test, rf_pred, alpha=0.5, color='teal', label='Random Forest Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
plt.title('Actual vs Predicted Salaries (10k Sample)')
plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.legend()
plt.show()